# Causal features and sampling

Build causal authentication and DNS features, then draw the time-stratified training sample (README sections 5 and 6).

In [ ]:
import sys
from pathlib import Path

import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

CONFIG = yaml.safe_load((ROOT / "configs" / "experiment.yaml").read_text())

## Causal features

Every feature uses only events with strictly earlier timestamps, so records sharing a second never see each other (README section 6). History starts at day 1; features are written from the first training day. Building every day takes roughly 1.5 to 2 hours; re-running requires deleting the output folder first.

In [ ]:
import json

import pyarrow.parquet as pq

from knightglass.features import build_features, label_rows

INTERIM = ROOT / CONFIG["data"]["interim_dir"]
PROCESSED = ROOT / CONFIG["data"]["processed_dir"]
FEATURES = PROCESSED / "features"

if (FEATURES / "_manifest.json").exists():
    feature_manifest = json.loads((FEATURES / "_manifest.json").read_text())
else:
    labels = label_rows(pq.read_table(INTERIM / "labels" / "redteam_matches.parquet"),
                        pq.read_table(INTERIM / "labels" / "possible_redteam.parquet"))
    feature_manifest = build_features(
        INTERIM / "auth", INTERIM / "dns", labels, FEATURES,
        CONFIG["features"]["windows_seconds"], CONFIG["dns"]["full_collection_from_day"],
        first_day=CONFIG["splits"]["train"][0])
{k: v for k, v in feature_manifest.items() if k != "rows_per_day"}

## Training samples

All known positives plus a time-stratified sample of unlabeled rows, capped at `sampling.unlabeled_cap` (README section 5.4). One sample for the training period, and one for the refit on training plus validation days (section 5.3). Possible red-team rows are excluded.

In [ ]:
import pandas as pd

from knightglass.sampling import time_stratified_reservoir

SAMPLES = PROCESSED / "samples"
sample_manifests = {}
for name, days in [("train", CONFIG["splits"]["train"]),
                   ("refit", CONFIG["refit_days"])]:
    _, sample_manifests[name] = time_stratified_reservoir(
        FEATURES, days, CONFIG["sampling"]["unlabeled_cap"], CONFIG["seed"],
        out_dir=SAMPLES, name=name)
pd.DataFrame({n: {k: v for k, v in m.items() if not isinstance(v, dict)}
              for n, m in sample_manifests.items()})